In [1]:
!pip install -q transformers datasets trl peft bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.4 MB/s eta 0:00:00


In [2]:
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import DPOTrainer
import random

# -------------------------------------------------------------------------
# 1. Configuration & T4 Scaling (QLoRA)
# -------------------------------------------------------------------------
# Using Llama-3-8B as specified in E2.7.
# NOTE: You must have accepted the Llama-3 terms on HuggingFace and logged in via `huggingface-cli login`
MODEL_ID = "meta-llama/Meta-Llama-3-8B"

# 4-bit Quantization Config to fit Llama-3 8B onto a 16GB T4 GPU
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

print("Loading tokenizer and model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)
model = prepare_model_for_kbit_training(model)

# LoRA Configuration for parameter-efficient fine-tuning
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
)
model = get_peft_model(model, peft_config)

# -------------------------------------------------------------------------
# 2. Dataset Preparation & Synthetic Preference Pairing (E2.7)
# -------------------------------------------------------------------------
# The paper states: "For DPO on verifiable tasks, construct preference pairs
# by pairing each correct completion with a randomly sampled incorrect completion".

print("Loading GSM8K dataset...")
dataset = load_dataset("gsm8k", "main")

def extract_answer(completion):
    """Extracts the final numeric answer from GSM8K ground truth."""
    return completion.split("#### ")[-1].strip()

def create_synthetic_dpo_pairs(dataset_split, num_samples=1000):
    """
    Simulates the rollout process by creating synthetic positive (correct)
    and negative (incorrect) pairs for the DPO format.
    """
    dpo_data = {
        "prompt": [],
        "chosen": [],
        "rejected": []
    }

    # Take a subset for reasonable Colab runtimes
    samples = list(dataset_split)[:num_samples]

    for i, item in enumerate(samples):
        prompt = f"Question: {item['question']}\nAnswer:"
        correct_completion = item['answer']

        # To strictly follow the paper: "pairing each correct completion with a
        # randomly sampled incorrect completion from the same batch".
        # We simulate this by grabbing a structurally valid but incorrect answer from another problem.
        random_incorrect_item = random.choice([x for j, x in enumerate(samples) if j != i])
        incorrect_completion = random_incorrect_item['answer']

        dpo_data["prompt"].append(prompt)
        dpo_data["chosen"].append(correct_completion)
        dpo_data["rejected"].append(incorrect_completion)

    return datasets.Dataset.from_dict(dpo_data)

print("Constructing synthetic preference pairs...")
train_dpo_dataset = create_synthetic_dpo_pairs(dataset['train'], num_samples=2000)
eval_dpo_dataset = create_synthetic_dpo_pairs(dataset['test'], num_samples=200)

# -------------------------------------------------------------------------
# 3. Training Setup (DPOTrainer)
# -------------------------------------------------------------------------
# The paper states DPO uses a standard binary cross-entropy loss over
# a fixed dataset of preference pairs[cite: 105, 107].

training_args = TrainingArguments(
    output_dir="./dpo_llama3_gsm8k",
    per_device_train_batch_size=2, # Keep low for T4 GPU
    gradient_accumulation_steps=4, # Increase to simulate larger batch size
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    max_steps=200, # Adjust based on your compute budget
    logging_steps=10,
    evaluation_strategy="steps",
    eval_steps=50,
    save_steps=50,
    fp16=True, # Use FP16 for T4 compatibility
    optim="paged_adamw_32bit",
    report_to="none"
)

# Initialize DPO Trainer.
# Note: trl's DPOTrainer automatically creates the reference model implicitly
# or uses the base un-fine-tuned model when ref_model=None.
dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=training_args,
    beta=0.1, # The beta parameter controls the KL penalty [cite: 49]
    train_dataset=train_dpo_dataset,
    eval_dataset=eval_dpo_dataset,
    tokenizer=tokenizer,
    max_prompt_length=256,
    max_length=512,
)

# -------------------------------------------------------------------------
# 4. Execute Experiment
# -------------------------------------------------------------------------
print("Starting DPO Training...")
dpo_trainer.train()

# Save final adapted model
dpo_trainer.save_model("./dpo_llama3_gsm8k_final")
print("Experiment complete. Model saved.")

Loading tokenizer and model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Meta-Llama-3-8B.
401 Client Error. (Request ID: Root=1-69fa6212-78a4979f0cea0ed123823134;ba7e00e1-c97d-437b-b737-5f6f89980eb2)

Cannot access gated repo for url https://huggingface.co/meta-llama/Meta-Llama-3-8B/resolve/main/config.json.
Access to model meta-llama/Meta-Llama-3-8B is restricted. You must have access to it and be authenticated to access it. Please log in.